# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: bc18755a-142a-44a5-987e-a1761aac3913
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session bc18755a-142a-44a5-987e-a1761aac3913 to get into ready status...


#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='orders')

In [3]:
dyf = dyf.apply_mapping([
    ("col0","string","order_id","string"),
    ("col1","string","customer_id","string"),
    ("col2","string","order_status","string"),
    ("col3","string","order_purchase_timestamp","timestamp"),
    ("col4","string","order_approved_at","timestamp"),
    ("col5","string","order_delivered_carrier_date","timestamp"),
    ("col6","string","order_delivered_customer_date","timestamp"),
    ("col7","string","order_estimated_delivery_date","timestamp")
])

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [4]:
df = dyf.toDF()

df_sample = df.limit(1000)
df_sample.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|                    NULL|               NULL|                        NULL|                         NULL|                         NULL|
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|  

In [6]:
# Data cleaning

# Step 1: remove all the rows whose order_id column is null
cleaned_df = df_sample.filter(df_sample.order_id.isNotNull())

# Step 2: standardize the column 'status', to remove spaces and makes sure all values are in lowercase
from pyspark.sql.functions import lower, regexp_replace, col

cleaned_df = cleaned_df.withColumn("order_status", lower(regexp_replace(col("order_status"), " ", "")))

# Step 3: remove all columns with duplicate order_id values 
cleaned_df = cleaned_df.dropDuplicates(["order_id"])

In [ ]:
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_df, glueContext, "orders_dyf")

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
# Write the cleaned data to S3 in Parquet format as output for the next job

s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/orders",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="orders"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(cleaned_dyf)